In [0]:
# =====================================================================
# proceso / 06_grants.py
# Task "grants" (última del job, ver Ejemplo 2). Aplica los permisos
# definidos en seguridad/01_grants.sql, sustituyendo ${catalogo}.
# =====================================================================

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("sql_file", "../seguridad/01_grants.sql")
dbutils.widgets.text("catalogo", "retail_medallion")
sql_file = dbutils.widgets.get("sql_file")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
with open(sql_file, "r", encoding="utf-8") as f:
    script = f.read()

script = script.replace("${catalogo}", catalogo)

statements = [s.strip() for s in script.split(";") if s.strip() and not s.strip().startswith("--")]

for stmt in statements:
    print(f"Ejecutando:\n{stmt}\n")
    spark.sql(stmt)

print(f"Grants aplicados: {len(statements)} sentencias ejecutadas sobre catalog '{catalogo}'.")

Ejecutando:
USE CATALOG retail_medallion

Ejecutando:
GRANT USE CATALOG ON CATALOG retail_medallion TO `retail_engineers`

Ejecutando:
GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA retail_medallion.bronze TO `retail_engineers`

Ejecutando:
GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA retail_medallion.silver TO `retail_engineers`

Ejecutando:
GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA retail_medallion.golden TO `retail_engineers`

Ejecutando:
GRANT USE CATALOG ON CATALOG retail_medallion TO `retail_readers`

Ejecutando:
GRANT USE SCHEMA, SELECT ON SCHEMA retail_medallion.golden TO `retail_readers`

Ejecutando:
REVOKE SELECT ON SCHEMA retail_medallion.bronze FROM `retail_readers`

Ejecutando:
REVOKE SELECT ON SCHEMA retail_medallion.silver FROM `retail_readers`

Ejecutando:
GRANT READ FILES ON EXTERNAL LOCATION `exlt-raw` TO `retail_engineers`

Ejecutando:
GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION `exlt-retail-data` TO `retail_engineers`

Eje

In [0]:
%sql
SELECT 'bronze' AS capa, COUNT(*) AS filas FROM retail_medallion.bronze.ecommerce_raw
UNION ALL
SELECT 'silver', COUNT(*) FROM retail_medallion.silver.ecommerce_daily_clean
UNION ALL
SELECT 'golden.marketing_roi', COUNT(*) FROM retail_medallion.golden.ecommerce_marketing_roi
UNION ALL
SELECT 'golden.agg_by_category (ecommerce)', COUNT(*) FROM retail_medallion.golden.agg_sales_by_category_month WHERE source_system = 'ecommerce';

capa,filas
golden.agg_by_category (ecommerce),165
bronze,1000
silver,1000
golden.marketing_roi,165
